# Finetuning LLAMA

## Installing the Libraries

In [ ]:
!pip install trl

In [ ]:
!pip install -U bitsandbytes

## Importing Hugging Face Access Token

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

## Logging in to Hugging Face

In [ ]:
from huggingface_hub import login

In [ ]:
login(HF_TOKEN)

## Logging in to WANDB

In [ ]:
import os
import wandb

In [ ]:
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login()

## Importing the Libraries

In [ ]:
import json
import torch
from datetime import datetime
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

## Importing the dataset

In [ ]:
FILENAME = "Finetune Dataset.jsonl"
TRAIN_EXAMPLES = 900

with open(FILENAME, "r", encoding = "utf-8") as f:
  lines = f.readlines()
  train_dataset = lines[:TRAIN_EXAMPLES]
  valid_dataset = lines[TRAIN_EXAMPLES:]

train_dataset = Dataset.from_list([json.loads(x) for x in train_dataset])
valid_dataset = Dataset.from_list([json.loads(x) for x in valid_dataset])

## Configurations

#### To Calculate Warm-up Steps:
```python
steps_per_epoch = math.ceil(TRAIN_EXAMPLES / BATCH_SIZE / GRADIENT_ACCUMULATION_STEPS)
total_training_steps = steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(total_training_steps * 0.01))

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_PREFIX = "LLAMA-Finetune"
HF_USERNAME = "username"
WANDB_ENTITY = "usernameteam"

TIMESTAMP = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_NAME = f"{PROJECT_PREFIX}-{TIMESTAMP}"
HUB_MODEL_NAME = f"{HF_USERNAME}/{PROJECT_NAME}"

EPOCHS = 3
BATCH_SIZE = 32
GRADIENT_ACCUMULATION = 1
MAX_SEQUENCE_LENGTH = 288
SAVE_STEPS = 10

# QLoRA Config

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.2

ATTENTION_LAYERS = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]

TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS

# Hyper-parameters

LEARNING_RATE = 1e-4
WARMUP_STEPS = 1
LR_SCHEDULER_TYPE = "cosine"
OPTIMIZER = "adamw_torch"
WEIGHT_DECAY = 0.001
MAX_GRAD_NORM = 0.3

## Starting Weight & Biases Session

In [ ]:
os.environ["WANDB_PROJECT"] = PROJECT_PREFIX
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
wandb.init(
    project = PROJECT_PREFIX,
    name = PROJECT_NAME,
    entity = WANDB_ENTITY,
)

## Quantization Configurations

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = "nf4"
)

## Loading the Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"

## Loading the model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map = "auto",
    quantization_config = quantization_config,
    dtype = torch.bfloat16
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
base_model.config.pad_token = tokenizer.pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
memory = base_model.get_memory_footprint() / 1e6
print(f"Memory Footprint: {memory:,.2f} MB")

Memory Footprint: 2,197.64 MB


## Configuring LoRA Parameters

In [ ]:
lora_parameters = LoraConfig(
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias = "none",
    task_type = "CAUSAL_LM"
)

## Configuring Training Parameters

In [ ]:
training_parameters = SFTConfig(
    output_dir = PROJECT_NAME,
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,

    learning_rate = LEARNING_RATE,
    optim = OPTIMIZER,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION,

    bf16 = True,
    warmup_steps = WARMUP_STEPS,
    lr_scheduler_type = LR_SCHEDULER_TYPE,

    report_to = "wandb",
    run_name = PROJECT_NAME,
    max_length = MAX_SEQUENCE_LENGTH,

    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    save_total_limit = 10,

    eval_strategy = "steps",
    eval_steps = SAVE_STEPS,

    push_to_hub = True,
    hub_model_id = HUB_MODEL_NAME,
    hub_private_repo = True
)

## Creating the Trainer

In [ ]:
finetuning = SFTTrainer(
    model = base_model,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    peft_config = lora_parameters,
    args = training_parameters
)

## Starting Finetuning

In [ ]:
finetuning.train()

finetuning.model.push_to_hub(PROJECT_NAME, private = True)
print(f"Saved model to hub: {PROJECT_NAME}")

## Statistics

#### Train Summary:

<sub>████████████████████████ 100% | 87/87 | 1:50:02 | Epoch 3/3</sub>

<br>

| Step | Training Loss | Validation Loss |  Entropy | Num Tokens | Mean Token Accuracy |
| ---: | ------------: | --------------: | -------: | ---------: | ------------------: |
|   10 |      0.494043 |        0.113875 | 0.106422 |     68,527 |            0.970634 |
|   20 |      0.056519 |        0.057026 | 0.048302 |    136,500 |            0.984164 |
|   30 |      0.032682 |        0.041884 | 0.030534 |    199,204 |            0.988312 |
|   40 |      0.024982 |        0.032319 | 0.031031 |    267,300 |            0.989696 |
|   50 |      0.020835 |        0.029931 | 0.027471 |    335,858 |            0.990019 |
|   60 |      0.022614 |        0.029174 | 0.027662 |    398,121 |            0.991579 |
|   70 |      0.018335 |        0.027050 | 0.026488 |    467,449 |            0.992358 |
|   80 |      0.016459 |        0.026668 | 0.026013 |    535,127 |            0.992514 |
|   87 |      0.016459 |        0.026678 | 0.026031 |    576,957 |            0.992704 |

<br>

#### Run Summary:

| Metric | Value |
|---|---:|
| `eval/entropy` | 0.02603 |
| `eval/loss` | 0.02668 |
| `eval/mean_token_accuracy` | 0.9927 |
| `eval/num_tokens` | 576,957 |
| `eval/runtime` | 69.1925 |
| `eval/samples_per_second` | 1.431 |
| `eval/steps_per_second` | 0.188 |
| `total_flos` | 1.1792343913709568e+16 |
| `train/entropy` | 0.01986 |
| `train/epoch` | 3 |

## Finishing WANDB Session

In [ ]:
wandb.finish()

## Testing the Fine-Tuned Model

In [ ]:
from peft import PeftModel
ADAPTER_MODEL = HF_USERNAME + "/LLAMA-Scribe"

In [ ]:
finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_MODEL
)

finetuned_model.eval()
print("Model loaded successfully!")

In [ ]:
prompt = "Extract the relevant fields from this OCR text, ignoring any unrelated text or artifacts:\n\"CITY HEALTH CLINIC\\nBILL NO: HC-48291\\nDATE: 08/MAR/2026\\nPATIENT: SARAH LIM\\nCONSULTATION $55.00\\nMEDICATION $25.00\\nGST (9%): $7.20\\nTOTAL: $87.20\\nREMARKS: FOLLOW-UP VISIT SUBSIDY APPLIED.\\nPAGE 1 OF 1\"\n"

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors = "pt"
).to(finetuned_model.device)

In [ ]:
with torch.no_grad():
  outputs = finetuned_model.generate(
      **inputs,
      max_new_tokens = 200,
      pad_token_id = tokenizer.eos_token_id
  )

In [ ]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens = True
)

In [ ]:
print(response[response.find("{"):])

{
  "doc_type": "medical_bill",
  "schema": {
    "Date_of_Visit": "08/MAR/2026",
    "Date_of_Visit_confidence_score": 98,
    "Amt": "87.20",
    "Amt_confidence_score": 99,
    "GST": "7.20",
    "GST_confidence_score": 95,
    "Remarks": "FOLLOW-UP VISIT SUBSIDY APPLIED.",
    "Remarks_confidence_score": 84
  }
}


## Testing Model using Pipeline

In [ ]:
prompt = """Extract the relevant fields from this OCR text, ignoring any unrelated text or artifacts:
"CITY HEALTH CLINIC
BILL NO: HC-48291
DATE: 08/MAR/2026
PATIENT: SARAH LIM
CONSULTATION $55.00
MEDICATION $25.00
GST (9%): $7.20
TOTAL: $87.20
REMARKS: FOLLOW-UP VISIT SUBSIDY APPLIED.
PAGE 1 OF 1"
"""

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model = finetuned_model,
    tokenizer = tokenizer
)

In [ ]:
result = generator(
    prompt,
    max_new_tokens = 200
)

In [ ]:
response = result[0]["generated_text"]
print(response[response.find("{"):])

{
  "doc_type": "medical_bill",
  "schema": {
    "Date_of_Visit": "08/MAR/2026",
    "Date_of_Visit_confidence_score": 98,
    "Amt": "87.20",
    "Amt_confidence_score": 99,
    "GST": "7.20",
    "GST_confidence_score": 95,
    "Remarks": "FOLLOW-UP VISIT SUBSIDY APPLIED.",
    "Remarks_confidence_score": 86
  }
}
